In [ ]:
import arcgis
import time
from arcgis.gis import GIS
from arcgis.gis import Item
from arcgis.apps.storymap import StoryMap

from typing import Set  # Import Set from typing
import re, json, csv

import pandas as pd
import os
import logging
import requests

# Set Pandas dataframe display options
pd.set_option('display.max_colwidth', None)
pd.set_option('display.max_columns',1000)

In [ ]:
agoNotebook = False
# Print the version of the arcgis module
print(f"Running ArcGIS API for Python version: {arcgis.__version__}")

# Define the GIS
if agoNotebook == False:
    import keyring
    service_name = "system" # Use the default local credential store
    success = False # Set initial state

    # Ask for the username
    while success == False:
        username_for_keyring = input("Enter your ArcGIS Online username:") # If you are using VS Code, the text input dialog box appears at the top of the window
        # Get the credential object
        credential = keyring.get_credential(service_name, username_for_keyring)
        # Check if the username is in the credential store
        if credential is None:
            print(f"'{username_for_keyring}' is not in the local system's credential store. Try another username.")
        # Retrieve the password, login and set the GIS portal
        else:
            password_from_keyring = keyring.get_password("system", username_for_keyring)
            portal_url = 'https://www.arcgis.com'  
            gis = GIS(portal_url, username=username_for_keyring, password=password_from_keyring)
            success = True
            # Print a success message with username and user's organization role
            print("Successfully logged in as: " + gis.properties.user.username, "(role: " + gis.properties.user.role + ")")
else:
    gis = GIS("home")

In [ ]:
classic_maptour_id = "20fd39888a444629bc8e40d9b6ac38cc"
classic_maptour_webmap = ""
classic_maptour_featureCollection = ""
classic_maptour_featureSet = ""

In [ ]:
# Retrieve the JSON data for the classic MapTour item
classic_item = gis.content.get(classic_maptour_id)
classic_json = classic_item.get_data()
# import pprint
# pprint.pprint(classic_item_json)  # Optional: inspect structure

# Find the webmap ID referenced in the item JSON (usually in 'values' > 'webmap')
webmap_id = None
if 'values' in classic_json and 'webmap' in classic_json['values']:
    webmap_id = classic_json['values']['webmap']
    print(f"Found webmap ID: {webmap_id}")
else:
    print("Webmap ID not found in item JSON.")

# Download the webmap's JSON data
classic_maptour_webmap = None
if webmap_id:
    webmap_item = gis.content.get(webmap_id)
    classic_maptour_webmap = webmap_item.get_data()
    # pprint.pprint(classic_maptour_webmap)  # Optional: inspect structure
else:
    print("Cannot retrieve webmap JSON without webmap ID.")

# Parse the webmap JSON to get the featureCollection and featureSet
classic_maptour_featureCollection = None
classic_maptour_featureSet = None
if classic_maptour_webmap:
    # Look for operationalLayers with type 'Feature Layer' or 'featureCollection'
    layers = classic_maptour_webmap.get('operationalLayers', [])
    for layer in layers:
        # Check for featureCollection
        if 'featureCollection' in layer:
            classic_maptour_featureCollection = layer['featureCollection']
            print("Found featureCollection in webmap.")
            # Check for featureSet inside featureCollection
            if 'layers' in classic_maptour_featureCollection:
                for fc_layer in classic_maptour_featureCollection['layers']:
                    if 'featureSet' in fc_layer:
                        classic_maptour_featureSet = fc_layer['featureSet']
                        print("Found featureSet in featureCollection.")
                        break
            break
    if not classic_maptour_featureCollection:
        print("No featureCollection found in webmap.")
    if not classic_maptour_featureSet:
        print("No featureSet found in featureCollection.")
else:
    print("Webmap JSON not loaded.")

In [ ]:
import os
import requests

# Create directories if they don't exist
pics_dir = "mapTourTest/pics"
thumbs_dir = "mapTourTest/thumbs"
os.makedirs(pics_dir, exist_ok=True)
os.makedirs(thumbs_dir, exist_ok=True)

# Download images from each feature in classic_maptour_featureSet
if classic_maptour_featureSet and "features" in classic_maptour_featureSet:
    for i, feature in enumerate(classic_maptour_featureSet["features"]):
        # Download main image
        pic_url = feature["attributes"].get("pic_url")
        if pic_url:
            pic_filename = os.path.join(pics_dir, f"pic_{i}.jpg")
            try:
                response = requests.get(pic_url, timeout=10)
                if response.status_code == 200:
                    with open(pic_filename, "wb") as f:
                        f.write(response.content)
                    print(f"Downloaded: {pic_filename}")
                else:
                    print(f"Failed to download {pic_url}: {response.status_code}")
            except Exception as e:
                print(f"Error downloading {pic_url}: {e}")

        # Download thumbnail image
        thumb_url = feature["attributes"].get("thumb_url")
        if thumb_url:
            thumb_filename = os.path.join(thumbs_dir, f"thumb_{i}.jpg")
            try:
                response = requests.get(thumb_url, timeout=10)
                if response.status_code == 200:
                    with open(thumb_filename, "wb") as f:
                        f.write(response.content)
                    print(f"Downloaded: {thumb_filename}")
                else:
                    print(f"Failed to download {thumb_url}: {response.status_code}")
            except Exception as e:
                print(f"Error downloading {thumb_url}: {e}")
else:
    print("classic_maptour_featureSet['features'] not found or empty.")

In [ ]:
import uuid
import json
import requests
from arcgis.apps.storymap import StoryMap
username = gis.properties.user.username

# Step 1: Create a new StoryMap draft (using ArcGIS API for Python)
storymap = StoryMap(gis=gis)
storymap_item = storymap.save(publish=True)
storymap_id = storymap_item.itemid
print(f"Created StoryMap with ID: {storymap_id}")

# Step 2: Upload images to the StoryMap resources endpoint using REST API
pics_dir = "mapTourTest/pics"
pic_files = [f for f in os.listdir(pics_dir) if f.lower().endswith(('.jpg','.jpeg','.png'))]
image_resource_map = {}  # Map pic filename to resourceId
portal_url = gis._portal.url if hasattr(gis, '_portal') else gis.url
# add_resource_url = f"{portal_url}/sharing/rest/content/items/{storymap_id}/addResource"
add_resource_url = f"https://www.arcgis.com/sharing/rest/content/users/{username}/items/{storymap_id}/addResources"
for file in pic_files:
    file_path = os.path.join(pics_dir, file)
    with open(file_path, "rb") as img_file:
        files = {"file": (file, img_file)}
        params = {
            "f": "json",
            "token": gis._con.token,
            "fileName": file
        }
        response = requests.post(add_resource_url, files=files, data=params)
        if response.status_code == 200 and response.json().get("success"):
            image_resource_map[file] = file  # Use filename as resourceId for mapping
            print(f"Uploaded resource: {file}")
        else:
            print(f"Failed to upload resource: {file}. Full response: {response.text}")


In [ ]:

# Step 3: Build StoryMap JSON mimicking graves-tour.json schema
from converter_json import StoryMapJSONBuilder
from storymap_json_schema import (
    create_tour_map_geometry,
    create_tour_map_node,
    create_tour_place,
    create_tour_node,
    set_cover_data,
    set_theme,
    generate_node_id,
)

def build_tourmap_json(feature_set, image_resource_map, title="(COPY) MapTour", theme_id="summit"):
    builder = StoryMapJSONBuilder(theme_id=theme_id)
    storymap_json = builder.get_json()

    # Generate node IDs for tour-map and tour
    tour_map_node_id = generate_node_id()
    tour_node_id = generate_node_id()

    # Build geometries for tour-map node
    geometries = {}
    places = []

    for i, feature in enumerate(feature_set["features"]):
        geom_id = str(uuid.uuid4())
        geom = create_tour_map_geometry(
            id=geom_id,
            long=feature["geometry"]["x"],
            lat=feature["geometry"]["y"],
            type="POINT_NUMBERED_TOUR"
        )
        geometries[geom_id] = geom

        # Title node
        title_text = feature["attributes"].get("name", "")
        title_node_id = builder.add_text(title_text, style="h2", alignment="start")

        # Description/content node(s)
        description_text = feature["attributes"].get("description", "")
        content_node_id = builder.add_text(description_text, style="paragraph", alignment="start")
        contents = [content_node_id]

        # Media node (image)
        pic_filename = f"pic_{i}.jpg"
        resource_name = image_resource_map.get(pic_filename, pic_filename)
        media_node_id = builder.add_image(resource_name)

        # Place node (references node IDs)
        place_id = generate_node_id()
        place = create_tour_place(
            id=place_id,
            feature_id=geom_id,
            contents=contents,
            media=media_node_id,
            title=title_node_id
        )
        places.append(place)

    # Create tour-map node
    builder.add_node(create_tour_map_node(
        geometries=geometries,
        mode="2d",
        basemap_type="name",
        basemap_value="worldImagery"
    ), parent_id=None)  # Add to root or as detached

    # Create tour node
    builder.add_node(create_tour_node(
        places=places,
        map_node_id=tour_map_node_id,
        accent_color="#f9f794",
        narrative_panel_position="start",
        narrative_panel_size="medium",
        tour_type="explorer",  # Use "explorer" and "list" for AGSM native
        subtype="list"
    ), parent_id=None)

    # Set cover and theme
    set_cover_data(storymap_json, title)
    set_theme(storymap_json, theme_id)

    return storymap_json

# Usage:
new_storymap_json = build_tourmap_json(classic_maptour_featureSet, image_resource_map)
with open("map-tour-generated.json", "w") as f:
    json.dump(new_storymap_json, f, indent=2)
print("StoryMap JSON generated and saved as map-tour-generated.json")

In [ ]:
import json
from storymap_json_schema import validate_node_against_schema

# Load the generated StoryMap JSON
with open("map-tour-generated.json", "r") as f:
    storymap_json = json.load(f)

# Collect all validation errors
all_errors = []
for node_id, node in storymap_json.get("nodes", {}).items():
    node_type = node.get("type")
    errors = validate_node_against_schema(node, node_type)
    if errors:
        print(f"Node {node_id} ({node_type}) errors:")
        for err in errors:
            print("  -", err)
        all_errors.extend(errors)

if not all_errors:
    print("No validation errors found in map-tour-generated.json!")
else:
    print(f"Total validation errors: {len(all_errors)}")